### Creating 1120 tests from test dataset

In [2]:
import pandas as pd

test_df = pd.read_parquet('test_data_with_predictions.parquet')

In [3]:
import numpy as np
hospital_df = (
    test_df
    .sort_values(["subject_id", "hadm_id", "day"])
    .groupby(["subject_id", "hadm_id"])
    .last()
    .reset_index()
)

threshold = 0.2158
hospital_df["risk_scaled"] = np.where(
    hospital_df["predicted_proba"] <= threshold,
    0.5 * hospital_df["predicted_proba"] / threshold,
    0.5 + 0.5 * (
        (hospital_df["predicted_proba"] - threshold)
        / (hospital_df["predicted_proba"].max() - threshold)
    )
)

def prediction_group(row):
    if row.true_class == 1 and row.predicted_class == 1:
        return "TP"
    elif row.true_class == 0 and row.predicted_class == 0:
        return "TN"
    elif row.true_class == 0 and row.predicted_class == 1:
        return "FP"
    else:
        return "FN"

hospital_df["group"] = hospital_df.apply(prediction_group, axis=1)
icd_cols = [c for c in hospital_df.columns if c.startswith("icd_")]
ccsr_cols = [c for c in hospital_df.columns if c.startswith("ccsr_")]
lab_cols = [c for c in hospital_df.columns if c.startswith("lab_")]

hospital_df["has_icd"] = hospital_df[icd_cols].sum(axis=1) > 0
hospital_df["has_ccsr"] = hospital_df[ccsr_cols].sum(axis=1) > 0
hospital_df["has_labs"] = hospital_df[lab_cols].notna().any(axis=1)
hospital_df["context_type"] = np.select(
    [
        hospital_df["has_icd"] & hospital_df["has_ccsr"] & hospital_df["has_labs"],

        (hospital_df["has_icd"] | hospital_df["has_ccsr"]) &
        (~hospital_df["has_labs"]),

        (~hospital_df["has_icd"]) &
        (~hospital_df["has_ccsr"]) &
        hospital_df["has_labs"],

        (~hospital_df["has_icd"]) &
        (~hospital_df["has_ccsr"]) &
        (~hospital_df["has_labs"]),
    ],
    [
        "full",
        "diagnoses_only",
        "labs_only",
        "empty",
    ],
    default="other"
)

In [4]:
selected = []

contexts = [
    "full",
    "diagnoses_only",
    "labs_only",
    "empty"
]

for group in ["TP", "TN", "FP", "FN"]:
    df_group = hospital_df[hospital_df["group"] == group]
    for context in contexts:
        df = df_group[df_group["context_type"] == context]
        if len(df) == 0:
            continue
        low = df.nsmallest(min(35, len(df)), "predicted_proba")
        high = df.nlargest(min(35, len(df)), "predicted_proba")

        selected.extend([low, high])

sample = (
    pd.concat(selected)
      .reset_index(drop=True)
)

In [5]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [6]:
def get_patient_full_data(subject_id, hadm_id, total_data, risk_score=False):
    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) &
        (total_data['hadm_id'] == hadm_id)
    ].sort_values("day")

    patient = patient_raw.iloc[-1].to_dict()

    json_context = {
        "demographics": {},
        "diagnoses": {},
        "laboratory_values": {},
        "clinical_indicators": {}
    }
    if risk_score:
        json_context = {
            "risk_score": {},
            "readmission": {},
            "demographics": {},
            "diagnoses": {},
            "laboratory_values": {},
            "clinical_indicators": {}
        }
        
        json_context["risk_score"] = round(patient.get("risk_scaled", None), 2)
        json_context["readmission"] = "True" if patient.get("true_class", None) == 1 else "False"

    json_context["demographics"]["gender"] = (
        "Male" if patient["gender_male"] == 1 else "Female"
    )
    json_context["demographics"]["length_of_stay"] = (patient["los"])

    race_cols = [c for c in total_data.columns if c.startswith("race_")]

    for col in race_cols:
        if patient.get(col) == 1:
            json_context["demographics"]["race"] = (
                col.replace("race_", "")
                .replace("_", " ")
                .title()
            )
            break

    insurance_cols = [col for col in total_data.columns if col.startswith('insurance_')]
    for col in insurance_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["insurance"] = col.replace('insurance_', '')
            break
    if 'insurance' not in json_context["demographics"]:
        json_context["demographics"]["insurance"] = 'Unknown'

    admission_type_cols = [col for col in total_data.columns if col.startswith('admission_type_')]
    for col in admission_type_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_type"] = col.replace('admission_type_', '').replace('_', ' ').title()
            break
    if 'admission_type' not in json_context["demographics"]:
        json_context["demographics"]["admission_type"] = 'Unknown'

    discharge_cols = [col for col in total_data.columns if col.startswith('discharge_location_')]
    for col in discharge_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["discharge_location"] = col.replace('discharge_location_', '').replace('_', ' ').title()
            break
    if 'discharge_location' not in json_context["demographics"]:
        json_context["demographics"]["discharge_location"] = 'Unknown'

    adm_location_cols = [col for col in total_data.columns if col.startswith('admission_location_')]
    for col in adm_location_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_location"] = col.replace('admission_location_', '').replace('_', ' ').title()
            break
    if 'admission_location' not in json_context["demographics"]:
        json_context["demographics"]["admission_location"] = 'Unknown'

    icd_list = []
    icd_cols = [c for c in total_data.columns if c.startswith("icd_")]
    for col in icd_cols:
        if patient.get(col) == 1:
            icd_list.append(map_feature_name(col))
    json_context["diagnoses"]["icd"] = icd_list

    ccsr_list = []
    ccsr_cols = [c for c in total_data.columns if c.startswith("ccsr_")]
    for col in ccsr_cols:
        if patient.get(col) == 1:
            ccsr_list.append(map_feature_name(col))
    json_context["diagnoses"]["ccsr"] = ccsr_list

    lab_cols = [
        c for c in total_data.columns
        if c.startswith("lab_") and c.endswith("_daily")]
    for col in lab_cols:
        value = patient.get(col)
        if pd.notna(value):
            json_context["laboratory_values"][map_feature_name(col)] = round(float(value), 2)

    other_features = [
        "num_diagnoses",
        "num_chronic",
        "comorbidity_score",
        "num_medications_daily",
        "prior_admissions_12mo",
        "cumulative_procedures",
        "cumulative_medications",
        "num_procedures_daily"
    ]

    for feat in other_features:
        value = patient.get(feat)
        if pd.notna(value):
            json_context["clinical_indicators"][feat] = float(value)

    return {
        "subject_id": int(subject_id),
        "hadm_id": int(hadm_id),
        "json_context": json_context
    }

In [7]:
import json
import numpy as np

patients_json = []

for _, row in sample.iterrows():
    patient = get_patient_full_data(
        row.subject_id,
        row.hadm_id,
        sample
    )
    patients_json.append(patient)

def convert_to_serializable(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, pd.Series):
        return obj.to_dict()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict('records')
    elif pd.isna(obj):
        return None
    else:
        return obj
    
with open("patients_sample.json", "w") as f:
    json.dump(
        convert_to_serializable(patients_json),
        f,
        indent=2,
        ensure_ascii=False
    )

### Examples for few-shot LLM test (20 examples)

In [8]:
sample_pairs = set(zip(sample.subject_id, sample.hadm_id))

fewshot_df = hospital_df[
    ~hospital_df.apply(
        lambda x: (x.subject_id, x.hadm_id) in sample_pairs,
        axis=1
    )
].copy()

fewshot_df = fewshot_df[
    fewshot_df["group"].isin(["TP", "TN"])
]

context_sizes = {
    "full": 4,
    "diagnoses_only": 2,
    "labs_only": 2,
    "empty": 2
}

fewshots = []
for group in ["TP", "TN"]:
    df_group = fewshot_df[fewshot_df["group"] == group]
    for context, n_examples in context_sizes.items():
        df = df_group[df_group["context_type"] == context]
        if len(df) == 0:
            continue

        quantiles = np.linspace(0.2, 0.8, n_examples)
        chosen_idx = []

        remaining = df.copy()
        
        for q in quantiles:
            target = remaining["predicted_proba"].quantile(q)
            idx = (remaining["predicted_proba"] - target).abs().idxmin()

            chosen_idx.append(idx)
            remaining = remaining.drop(idx)

        fewshots.append(df.loc[chosen_idx])

fewshot_examples = pd.concat(fewshots, ignore_index=True)

In [9]:
fewshot_examples

,subject_id,hadm_id,dischtime,los,age,day,current_date,num_diagnoses,num_chronic,ccsr_FAC021,...,predicted_proba,predicted_class,true_class,correct_prediction,risk_scaled,group,has_icd,has_ccsr,has_labs,context_type
0,17733544,29849977,2173-11-26,22,71,22,2173-11-24,858,396,0,...,0.262226,1,1,True,0.546389,TP,True,True,True,full
1,16277550,28885664,2181-12-09,4,61,4,2181-12-07,80,44,0,...,0.316344,1,1,True,0.600465,TP,True,True,True,full
2,14170015,23452761,2164-02-16,3,45,3,2164-02-14,30,18,0,...,0.384854,1,1,True,0.668921,TP,True,True,True,full
3,11582633,26534734,2138-06-07,1,55,1,2138-06-06,8,5,0,...,0.470784,1,1,True,0.754783,TP,True,True,True,full
4,18451972,26789515,2158-09-13,1,32,1,2158-09-11,7,1,1,...,0.275746,1,1,True,0.559899,TP,True,True,False,diagnoses_only
5,19620469,26950375,2184-10-02,2,42,2,2184-10-01,42,24,1,...,0.509620,1,1,True,0.793589,TP,True,True,False,diagnoses_only
6,17230816,26193652,2188-09-30,3,43,3,2188-09-29,33,9,0,...,0.286802,1,1,True,0.570946,TP,False,False,True,labs_only
7,13050559,24291663,2147-05-09,4,22,4,2147-05-08,36,16,0,...,0.451442,1,1,True,0.735456,TP,False,False,True,labs_only
8,13190051,29346610,2180-02-24,1,20,1,2180-02-22,6,1,0,...,0.322067,1,1,True,0.606183,TP,False,False,False,empty
9,12566043,25292821,2125-02-24,2,40,2,2125-02-22,2,0,0,...,0.393546,1,1,True,0.677607,TP,False,False,False,empty


In [10]:
fewshot = []

for _, row in fewshot_examples.iterrows():
    patient = get_patient_full_data(
        row.subject_id,
        row.hadm_id,
        fewshot_examples,
        risk_score=True
    )
    fewshot.append(patient)

with open("fewshot.json", "w") as f:
    json.dump(
        convert_to_serializable(fewshot),
        f,
        indent=2,
        ensure_ascii=False
    )

### Comparing one-shot MedGemma answers to LightGBM

Example where LLM gave risk_score: null

In [11]:
import json
with open('llm_risk_scores2.json', 'r', encoding='utf-8') as f:
    llm_risk_scores = json.load(f)

for llm_risk in llm_risk_scores:
    subject_id = llm_risk['subject_id']
    hadm_id = llm_risk['hadm_id']
    risk_score_llm = llm_risk['explanation']
    risk_score_llm = json.loads(risk_score_llm.strip('`')[5:])['risk_score']
    cur_hosp = sample[(sample['subject_id'] == subject_id) & (sample['hadm_id'] == hadm_id)]
    risk_score_ml = cur_hosp['risk_scaled'].values[0]

    if risk_score_llm == None:
        risk_score_llm = 0
        print(subject_id)
        print(hadm_id)
        print(risk_score_ml)
        print(cur_hosp)

10326996
20854950
0.8864995300654734
     subject_id   hadm_id   dischtime  los  age  day current_date  \
180    10326996  20854950  2138-04-05    3   32    3   2138-04-04   

     num_diagnoses  num_chronic  ccsr_FAC021  ...  predicted_proba  \
180              3            0            0  ...         0.602604   

     predicted_class  true_class  correct_prediction  risk_scaled  group  \
180                1           1                True       0.8865     TP   

     has_icd  has_ccsr  has_labs  context_type  
180    False     False      True     labs_only  

[1 rows x 148 columns]


In [12]:
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error
results = []

for llm_risk in llm_risk_scores:
    subject_id = llm_risk["subject_id"]
    hadm_id = llm_risk["hadm_id"]

    risk_llm = json.loads(
        llm_risk["explanation"].strip("`")[5:]
    )["risk_score"]

    row = sample[
        (sample.subject_id == subject_id) &
        (sample.hadm_id == hadm_id)
    ].iloc[0]

    results.append({
        "subject_id": subject_id,
        "hadm_id": hadm_id,
        "risk_ml": row["risk_scaled"],
        "risk_llm": risk_llm if risk_llm else 0,
        "true_class": row["true_class"],
        "group": row["group"],
        "context_type": row["context_type"],
        "pred_ml": row["predicted_class"]
    })

comparison = pd.DataFrame(results)
comparison["risk_diff"] = comparison.risk_llm-comparison.risk_ml
comparison["pred_llm"] = (comparison["risk_llm"] >= 0.5).astype(int)

pear = pearsonr(comparison.risk_ml, comparison.risk_llm)
spear = spearmanr(comparison.risk_ml, comparison.risk_llm)
mae = mean_absolute_error(
    comparison.risk_ml,
    comparison.risk_llm
)
print('Total statisctics:')
print(f"Pearson: {pear[0]}")
print(f"Spearman: {spear[0]}")
print(f"MAE: {mae}")

Total statisctics:
Pearson: 0.5975799694245149
Spearman: 0.5570487138290506
MAE: 0.2061815071888506


In [13]:
from sklearn.metrics import accuracy_score
def evaluate_subset(df, group_name, context_name):
    result = {
        "group": group_name,
        "context": context_name,
        "n": len(df),

        "pearson": pearsonr(df["risk_ml"], df["risk_llm"])[0],
        "spearman": spearmanr(df["risk_ml"], df["risk_llm"])[0],
        "mae": mean_absolute_error(df["risk_ml"], df["risk_llm"]),
        "mean_difference": df["risk_diff"].mean(),

        "accuracy_llm": accuracy_score(
            df["true_class"],
            df["pred_llm"]
        )
    }

    return result

In [14]:
results = []

groups = ["TP", "TN", "FP", "FN"]
contexts = ["full", "diagnoses_only", "labs_only", "empty"]

for g in groups:
    df_group = comparison[comparison.group == g]
    for c in contexts:
        df = df_group[df_group.context_type == c]
        res = evaluate_subset(df, g, c)

        if res is not None:
            results.append(res)
            
for g in groups:
    df = comparison[comparison.group == g]
    res = evaluate_subset(df, g, "ALL")
    if res is not None:
        results.append(res)

summary = pd.DataFrame(results)
summary

,group,context,n,pearson,spearman,mae,mean_difference,accuracy_llm
0,TP,full,70,0.246011,0.098598,0.245111,-0.142714,0.800000
1,TP,diagnoses_only,70,0.543125,0.617471,0.260307,-0.211172,0.685714
2,TP,labs_only,70,0.292977,0.329765,0.255343,-0.232902,0.414286
3,TP,empty,70,0.733770,0.622218,0.318665,-0.309733,0.371429
4,TN,full,70,0.638693,0.650879,0.174979,0.113787,0.685714
5,TN,diagnoses_only,70,0.722420,0.629849,0.150189,0.070107,0.685714
6,TN,labs_only,70,0.585659,0.631148,0.159601,-0.008398,0.928571
7,TN,empty,70,0.333542,0.207426,0.181799,-0.016922,0.914286
8,FP,full,70,0.311501,0.247059,0.220542,-0.117773,0.214286
9,FP,diagnoses_only,70,0.595983,0.507754,0.202103,-0.159546,0.342857


### Comparing few-shot MedGemma answers to LightGBM

In [15]:
import json
with open('llm_risk_scores_fsh.json', 'r', encoding='utf-8') as f:
    llm_risk_scores = json.load(f)

for llm_risk in llm_risk_scores:
    subject_id = llm_risk['subject_id']
    hadm_id = llm_risk['hadm_id']
    risk_score_llm = llm_risk['explanation']
    risk_score_llm = json.loads(risk_score_llm.strip('`')[5:])['risk_score']
    cur_hosp = sample[(sample['subject_id'] == subject_id) & (sample['hadm_id'] == hadm_id)]
    risk_score_ml = cur_hosp['risk_scaled'].values[0]

    if risk_score_llm == None:
        risk_score_llm = 0
        print(subject_id)
        print(hadm_id)
        print(risk_score_ml)
        print(cur_hosp)

In [16]:
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error
results = []

for llm_risk in llm_risk_scores:
    subject_id = llm_risk["subject_id"]
    hadm_id = llm_risk["hadm_id"]

    risk_llm = json.loads(
        llm_risk["explanation"].strip("`")[5:]
    )["risk_score"]

    row = sample[
        (sample.subject_id == subject_id) &
        (sample.hadm_id == hadm_id)
    ].iloc[0]

    results.append({
        "subject_id": subject_id,
        "hadm_id": hadm_id,
        "risk_ml": row["risk_scaled"],
        "risk_llm": risk_llm if risk_llm else 0,
        "true_class": row["true_class"],
        "group": row["group"],
        "context_type": row["context_type"],
        "pred_ml": row["predicted_class"]
    })

comparison = pd.DataFrame(results)
comparison["risk_diff"] = comparison.risk_llm-comparison.risk_ml
comparison["pred_llm"] = (comparison["risk_llm"] >= 0.5).astype(int)

pear = pearsonr(comparison.risk_ml, comparison.risk_llm)
spear = spearmanr(comparison.risk_ml, comparison.risk_llm)
mae = mean_absolute_error(
    comparison.risk_ml,
    comparison.risk_llm
)
print('Total statisctics:')
print(f"Pearson: {pear[0]}")
print(f"Spearman: {spear[0]}")
print(f"MAE: {mae}")

Total statisctics:
Pearson: 0.6586596529433972
Spearman: 0.5931532976945296
MAE: 0.2143445405055775


In [17]:
results = []

groups = ["TP", "TN", "FP", "FN"]
contexts = ["full", "diagnoses_only", "labs_only", "empty"]

for g in groups:
    df_group = comparison[comparison.group == g]
    for c in contexts:
        df = df_group[df_group.context_type == c]
        res = evaluate_subset(df, g, c)

        if res is not None:
            results.append(res)
            
for g in groups:
    df = comparison[comparison.group == g]
    res = evaluate_subset(df, g, "ALL")
    if res is not None:
        results.append(res)

summary = pd.DataFrame(results)
summary

,group,context,n,pearson,spearman,mae,mean_difference,accuracy_llm
0,TP,full,70,0.577177,0.476276,0.215184,-0.201000,0.757143
1,TP,diagnoses_only,70,0.727468,0.752971,0.315743,-0.315743,0.400000
2,TP,labs_only,70,0.476561,0.545309,0.283001,-0.278616,0.371429
3,TP,empty,70,0.659147,0.538192,0.380033,-0.370590,0.285714
4,TN,full,70,0.822567,0.749835,0.125255,0.080072,0.757143
5,TN,diagnoses_only,70,0.701933,0.644224,0.129610,-0.037607,0.985714
6,TN,labs_only,70,0.815749,0.819175,0.126984,-0.054826,0.971429
7,TN,empty,70,0.301243,0.150170,0.182226,-0.134350,0.985714
8,FP,full,70,0.569547,0.446530,0.201612,-0.181630,0.300000
9,FP,diagnoses_only,70,0.671528,0.550636,0.299400,-0.296546,0.614286
